In [2]:
import os
import pandas as pd

dataset = "/kaggle/input/big-startup-secsees-fail-dataset-from-crunchbase/big_startup_secsees_dataset.csv"
if not os.path.exists(dataset):
    dataset = "big_startup_secsees_dataset.csv"
df_original = pd.read_csv(dataset)

In [3]:
df_original.head()

,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
0,/organization/-fame,#fame,http://livfame.com,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaN,2015-01-05,2015-01-05
1,/organization/-qounter,:Qounter,http://www.qounter.com,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
2,/organization/-the-one-of-them-inc-,"(THE) ONE of THEM,Inc.",http://oneofthem.jp,Apps|Games|Mobile,3406878,operating,NaN,NaN,NaN,NaN,1,NaN,2014-01-30,2014-01-30
3,/organization/0-6-com,0-6.com,http://www.0-6.com,Curated Web,2000000,operating,CHN,22,Beijing,Beijing,1,2007-01-01,2008-03-19,2008-03-19
4,/organization/004-technologies,004 Technologies,http://004gmbh.de/en/004-interact,Software,-,operating,USA,IL,"Springfield, Illinois",Champaign,1,2010-01-01,2014-07-24,2014-07-24


In [4]:
df = df_original[~(df_original['status'] == 'operating')]

In [5]:
df.shape

(13334, 14)

In [6]:
df['success'] = df['status'].map({
    'ipo': 1,
    'acquired': 1,
    'closed': 0
})

In [7]:
df = df.drop(columns=['permalink', 'name', 'homepage_url', 'status', 'state_code', 'region', 'city'])

In [8]:
df.head()

,category_list,funding_total_usd,country_code,funding_rounds,founded_at,first_funding_at,last_funding_at,success
15,Apps|Cable|Distribution|Software,5000000,USA,1,2012-03-01,2015-03-17,2015-03-17,1
20,Art|E-Commerce|Marketplaces,500000,USA,1,2009-01-01,2009-05-15,2009-05-15,1
23,Curated Web,2535000,USA,2,2010-07-01,2010-01-01,2011-02-16,1
31,Analytics,1250000,USA,2,2011-09-16,2011-11-02,2011-11-30,1
32,Software,35000000,USA,1,2000-01-01,2010-03-08,2010-03-08,1


In [9]:
df.isna().sum()

category_list        1086
funding_total_usd       0
country_code         1991
funding_rounds          0
founded_at           3732
first_funding_at        2
last_funding_at         0
success                 0
dtype: int64

In [10]:
df = df.dropna(subset=['founded_at', 'first_funding_at'])

In [11]:
df.isna().sum()

category_list        434
funding_total_usd      0
country_code         958
funding_rounds         0
founded_at             0
first_funding_at       0
last_funding_at        0
success                0
dtype: int64

In [12]:
cols = ['category_list', 'country_code']
df[cols] = df[cols].fillna('Unknown')

for c in cols:
    print(df[c].value_counts().head(10))

category_list
Software               692
Biotechnology          506
Unknown                434
Curated Web            264
Mobile                 207
Enterprise Software    192
E-Commerce             158
Advertising            152
Games                  146
Semiconductors         146
Name: count, dtype: int64
country_code
USA        6275
Unknown     958
GBR         394
CAN         289
ISR         171
FRA         144
DEU         138
CHN         120
RUS         112
IND         105
Name: count, dtype: int64


In [13]:
df.dtypes

category_list          str
funding_total_usd      str
country_code           str
funding_rounds       int64
founded_at             str
first_funding_at       str
last_funding_at        str
success              int64
dtype: object

In [14]:
mask = pd.to_numeric(df['funding_total_usd'], errors='coerce').isna()
df.loc[mask, 'funding_total_usd'].value_counts()

funding_total_usd
-    1419
Name: count, dtype: int64

In [15]:
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')

In [16]:
df['funding_total_usd'].value_counts(dropna=False)

funding_total_usd
NaN            1419
1000000.0       152
500000.0        141
2000000.0       121
100000.0        119
               ... 
3384225.0         1
2257464.0         1
3805520.0         1
866550786.0       1
15419877.0        1
Name: count, Length: 3643, dtype: int64

In [17]:
df['funding_missing'] = df['funding_total_usd'].isna()

In [18]:
dates = ['founded_at', 'first_funding_at', 'last_funding_at']
df[dates] = df[dates].apply(pd.to_datetime, errors='coerce')

In [19]:
df.dtypes

category_list                   str
funding_total_usd           float64
country_code                    str
funding_rounds                int64
founded_at           datetime64[us]
first_funding_at     datetime64[us]
last_funding_at      datetime64[us]
success                       int64
funding_missing                bool
dtype: object

In [20]:
df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days
df['funding_duration'] = (df['last_funding_at'] - df['first_funding_at']).dt.days

In [21]:
df = df.drop(columns=['founded_at', 'first_funding_at', 'last_funding_at'])

In [22]:
days = ['days_to_first_funding', 'funding_duration']

summary = pd.DataFrame({
    'null': df[days].isna().sum(),
    'negative': (df[days] < 0).sum(),
    'zero': (df[days] == 0).sum()
})

display(summary)
print(f"Companies with a single funding round: {(df['funding_rounds'] == 1).sum()}")

,null,negative,zero
days_to_first_funding,0,763,663
funding_duration,0,0,5311


Companies with a single funding round: 5280


In [23]:
mask = (df['funding_duration'] == 0) & (df['funding_rounds'] > 1)
comparison = df.loc[mask, ['funding_duration', 'funding_rounds']]
display(comparison)
print(f'Companies with funding duration "zero" but more than one funding round: {mask.sum()}')

,funding_duration,funding_rounds
133,0,2
1689,0,2
2825,0,2
4128,0,2
4917,0,2
5968,0,2
7258,0,2
8345,0,2
11246,0,2
11411,0,2


Companies with funding duration "zero" but more than one funding round: 31


In [24]:
df = df[df['days_to_first_funding'] >= 0]

In [25]:
df['category'] = df['category_list'].str.split('|').str[0]
df = df.drop(columns='category_list')

In [26]:
df.dtypes

funding_total_usd        float64
country_code                 str
funding_rounds             int64
success                    int64
funding_missing             bool
days_to_first_funding      int64
funding_duration           int64
category                  object
dtype: object

In [27]:
X = df.drop(columns='success')
y = df['success']